# Scale up the PROVEN backbone on Colab — thousands of GAMUS tiles

Colab version of kaggle_scale_train.ipynb -- same plan, same architecture (the
one already in production, NOT the experimental HTC head), same reasoning:
the current checkpoint used only 400 of the 3,837 GAMUS tiles actually
available and it already works (MAE 1.444m, cross-domain 3.063m per-building
RMSE on DFC2019). This scales that same approach up to 2,500 tiles.

**Before running:**
1. Runtime -> Change runtime type -> **T4 GPU** -> Save.
2. Click the key icon in the left sidebar (Secrets) -> add a secret named
   `GITHUB_TOKEN` -> paste the same GitHub PAT you created for Kaggle (repo
   scope) -> toggle **Notebook access: On** for this notebook.

No dataset upload needed -- GAMUS downloads directly over HTTP.

**Colab-specific caveat**: a free Colab session disconnects after ~12 hours,
or sooner if idle/the browser tab closes. This run is sized (2,500 tiles,
15 epochs) to finish in an estimated 3-4 hours on a T4, comfortably inside
that limit -- but keep the tab open (an idle Colab tab can disconnect around
90 minutes of inactivity even within the 12-hour cap). If it disconnects
partway, `train_height.py` still saves the best checkpoint seen so far to
`/content/depthwizard/checkpoints/` before each improvement, so a partial
run is not a wasted one -- rerun from Cell 1 and it starts over, but you can
also lower `--epochs` in Cell 4 and just accept whatever it reaches.

In [ ]:
# Cell 1: clone the private repo using the GitHub token secret
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')

!rm -rf /content/depthwizard
!git clone --depth 1 https://{token}@github.com/jawaharananth/depthwizard.git /content/depthwizard
%cd /content/depthwizard

In [ ]:
# Cell 2: dependencies. torch is already on Colab's GPU image with CUDA wired up;
# only the project-specific packages need installing.
!pip install -q transformers h5py rasterio opencv-python-headless

In [ ]:
# Cell 3: confirm the GPU actually attached before spending download time for nothing.
import torch
assert torch.cuda.is_available(), "No GPU visible -- check Runtime > Change runtime type > T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# Cell 4: download a real 2,500-tile GAMUS subset (of 3,837 available) plus a 400-tile val
# set -- 6x the 400 tiles the current production checkpoint was trained on. `--limit`
# spreads the sample across the tile list (both PHL and DC), not a prefix of one city.
!python gamus.py download --split train --limit 2500
!python gamus.py download --split val --limit 400

In [ ]:
# Cell 5: the real run. Same architecture as the currently-licensed checkpoint (plain
# regression, NOT --htc). batch=8, accum=2 sized for a T4's ~15GB VRAM at crop=518 with
# the Base backbone -- drop --batch if this OOMs.
# 2,500 tiles x 4 crops/tile = 10,000 crops/epoch; ~15-20 crops/s on a T4 puts one epoch at
# roughly 10-15 min, so 15 epochs is a ~3-4 hour run. The script keeps only the best-MAE
# checkpoint seen so far, so it can be interrupted after any completed epoch without losing
# the best result reached up to that point.
!python train_height.py \
    --epochs 15 --batch 8 --accum 2 --crops-per-tile 4 --workers 2 \
    --out /content/depthwizard/checkpoints

In [ ]:
# Cell 6: package for download. Colab has no persistent /kaggle/working/ equivalent --
# this zip lives in the Colab VM's disk and disappears when the session ends, so download
# it via the Files sidebar (folder icon) or the download line this prints, right after
# Cell 5 finishes, before closing the tab.
!zip -r /content/metric_checkpoint.zip /content/depthwizard/checkpoints
from google.colab import files
files.download('/content/metric_checkpoint.zip')

## After downloading

Compare `training_report.json`'s `best_mae_m` against the current local
checkpoint's `checkpoints/height_best/metric_ok.json`
(`finetuned_mae_m_no_affine: 1.444`). If the new zip's `height_best/` folder
contains a `metric_ok.json` AND its `finetuned_mae_m_no_affine` beats 1.444,
copy that whole `height_best/` folder over the local one --
`depth_model.py`'s `DepthBackbone.metric_available()` picks it up
automatically. If `metric_ok.json` is absent, this run did not beat its own
freshly-measured baseline -- do not copy it over, same licensing rule as
always.